# Backtrack Effects - Finding Items by Buff Type

This notebook shows how to:
1. Find all items that provide a specific buff type
2. Search for items by building target
3. Find items with specific effects (e.g., productivity, additional output)
4. Build a reverse index

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter
from collections import defaultdict

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts

# Set language
LANGUAGE = "english"
texts.converter = StandardTextConverter(LANGUAGE)

print("Assets loaded!")

Assets loaded!


## Find Items by Buff Type

Search for items that provide productivity upgrades:

In [2]:
def find_items_with_productivity_buff(max_results=10):
    """Find items that provide productivity buffs."""
    items_template = assets.templates["Item"]
    results = []
    
    for item in items_template.assets:
        if not hasattr(item, 'Effect'):
            continue
        
        # Use buff_ui to check for productivity
        for buff_ui in item.buff_ui:
            buff_text = buff_ui.text()
            if 'productivity' in buff_text.lower():
                results.append((item.text(), buff_ui.value, item.guid))
                break
        
        if len(results) >= max_results:
            break
    
    return results

# Find items with productivity buffs
productivity_items = find_items_with_productivity_buff(max_results=15)

print("Items with Productivity Buffs:\n")
for name, value, guid in productivity_items:
    print(f"  • {name:50s} {value} (GUID: {guid})")

Items with Productivity Buffs:

  • Elephant Handler                                   +20% (GUID: 44431)
  • Servia Bellia, Lily of the Coast                   +25% (GUID: 95403)
  • Virtuous Volunteer                                 +15% (GUID: 96819)
  • Apion Mochthos Apicius, of Apeiron Appetite        +25% (GUID: 96817)
  • Fideus, Loyal Hound                                +35% (GUID: 106846)
  • Netter                                             +10% (GUID: 44075)
  • Oat Roller                                         +10% (GUID: 44076)
  • Mudlark                                            +10% (GUID: 44077)
  • Sailer                                             +10% (GUID: 44078)
  • Twiner                                             +10% (GUID: 44079)
  • Diviner of Dough                                   +10% (GUID: 44080)
  • Seamster                                           +10% (GUID: 44083)
  • Piscine Docent                                     +10% (GUID: 44085)
  • T

## Find Items by Target Building

Search for items that affect a specific building:

In [3]:
def find_items_affecting_building(building_guid, max_results=10):
    """Find all items that affect a specific building."""
    items_template = assets.templates["Item"]
    results = []
    
    target_building = assets.get(building_guid)
    if not target_building:
        return results
    
    for item in items_template.assets:
        if not hasattr(item, 'Effect'):
            continue
        
        # Check if this item targets our building
        targets = item.Effect.Targets
        if not targets:
            continue
        
        for target_entry in targets:
            target_asset = target_entry.GUID()
            if target_asset and target_asset.guid == building_guid:
                results.append((item.text(), item.guid))
                break
        
        if len(results) >= max_results:
            break
    
    return results

# Example: Find items that affect a specific building
if "Production" in assets.templates.elements:
    example_building = list(assets.templates["Production"].assets)[0]
    building_name = example_building.text()
    
    print(f"Finding items that affect: {building_name}\n")
    
    affecting_items = find_items_affecting_building(example_building.guid, max_results=10)
    
    if affecting_items:
        print(f"Found {len(affecting_items)} items:\n")
        for name, guid in affecting_items:
            print(f"  • {name} (GUID: {guid})")
    else:
        print("No items found (or no specific targets defined)")

Finding items that affect: Fishing Hut

Found 1 items:

  • Reangler (GUID: 106636)


## Find Items with Additional Output

Search for items that provide additional product output:

In [4]:
def find_items_with_additional_output(max_results=15):
    """Find items that provide additional output."""
    items_template = assets.templates["Item"]
    results = []
    
    for item in items_template.assets:
        # Use buff_ui to find additional output
        for buff_ui in item.buff_ui:
            buff_text = buff_ui.text()
            # Check for "Additional" or specific patterns
            if 'additional' in buff_text.lower() or 'additionally' in buff_text.lower():
                results.append((item.text(), buff_text, buff_ui.value, item.guid))
                break
        
        if len(results) >= max_results:
            break
    
    return results

# Find items with additional output
additional_output_items = find_items_with_additional_output(max_results=15)

print("Items with Additional Output:\n")
for name, buff_text, value, guid in additional_output_items:
    print(f"  • {name:45s}")
    print(f"    {buff_text}: {value}")

Items with Additional Output:



In [5]:
def find_items_by_rarity_and_buff(rarity="Legendary", buff_keyword="Productivity", max_results=10):
    """Find items matching both rarity and buff type."""
    items_template = assets.templates["Item"]
    results = []
    
    for item in items_template.assets:
        # Check rarity
        if hasattr(item.Item, 'Rarity'):
            item_rarity = item.Item.Rarity()
            if item_rarity != rarity:
                continue
        else:
            continue
        
        # Check buffs
        buff_list = item.buff_ui
        for buff_ui in buff_list:
            buff_text = buff_ui.text.values.get(LANGUAGE) if hasattr(buff_ui.text, 'values') else str(buff_ui.text)
            
            if buff_keyword.lower() in buff_text.lower():
                item_name = item.text.values.get(LANGUAGE, 'N/A') if item.text else 'N/A'
                results.append((item_name, buff_text, buff_ui.value, item.guid))
                break
        
        if len(results) >= max_results:
            break
    
    return results

# Find legendary items with productivity buff
legendary_productivity = find_items_by_rarity_and_buff("Legendary", "Productivity", max_results=10)

print("Legendary Items with Productivity Buff:\n")
for name, buff_text, value, guid in legendary_productivity:
    print(f"  • {name:45s}")
    print(f"    {buff_text}: {value}")
    print()

Legendary Items with Productivity Buff:



## Build a Reverse Index

Create a comprehensive index mapping buff types to items:

In [6]:
def build_buff_type_index(limit=None):
    """Build an index of buff types to items."""
    buff_type_to_items = defaultdict(list)
    items_template = assets.templates["Item"]
    
    for i, item in enumerate(items_template.assets):
        if limit and i >= limit:
            break
        
        # Get all buff_ui from the item
        for buff_ui in item.buff_ui:
            # Use the text as the buff type key
            buff_text = buff_ui.text()
            
            # Store item info
            buff_type_to_items[buff_text].append({
                'name': item.text(),
                'guid': item.guid,
                'value': buff_ui.value
            })
    
    return buff_type_to_items

# Build index for first 100 items
print("Building buff type index (this may take a moment)...\n")
buff_index = build_buff_type_index(limit=100)

print(f"Found {len(buff_index)} unique buff types\n")
print("Top 10 most common buff types:\n")

# Sort by number of items
sorted_buffs = sorted(buff_index.items(), key=lambda x: len(x[1]), reverse=True)

for i, (buff_type, items) in enumerate(sorted_buffs[:10], 1):
    print(f"{i:2d}. {buff_type:40s}: {len(items):3d} items")

Building buff type index (this may take a moment)...

Found 34 unique buff types

Top 10 most common buff types:

 1. Workforce Needed                        :  20 items
 2. Upkeep Cost                             :  18 items
 3. Belief Area Effect                      :  14 items
 4. Prestige Area Effect                    :  13 items
 5. Income Area Effect                      :  12 items
 6. Knowledge Area Effect                   :   8 items
 7. Prestige                                :   5 items
 8. Productivity                            :   5 items
 9. Trade Prices                            :   3 items
10. Income                                  :   3 items


## Query the Buff Index

Use the index to quickly find items with specific buff types:

In [7]:
# Search for a specific buff type
search_term = "Productivity"  # Change this to search for different buffs

print(f"Searching for items with '{search_term}' buff:\n")

found = False
for buff_type, items in buff_index.items():
    if search_term.lower() in buff_type.lower():
        found = True
        print(f"\nBuff Type: {buff_type}")
        print(f"Found {len(items)} items:\n")
        
        for item_info in items[:5]:  # Show first 5
            print(f"  • {item_info['name']:45s} {item_info['value']}")
        
        if len(items) > 5:
            print(f"  ... and {len(items) - 5} more")

if not found:
    print(f"No items found with '{search_term}' buff")
    print(f"\nAvailable buff types: {', '.join(list(buff_index.keys())[:10])}...")

Searching for items with 'Productivity' buff:


Buff Type: Productivity
Found 5 items:

  • Elephant Handler                              +20%
  • Servia Bellia, Lily of the Coast              +25%
  • Virtuous Volunteer                            +15%
  • Apion Mochthos Apicius, of Apeiron Appetite   +25%
  • Fideus, Loyal Hound                           +35%


## Find Items by Rarity and Buff Type

Combine multiple filters to find specific items:

In [8]:
def find_items_by_rarity_and_buff(rarity="Legendary", buff_keyword="Productivity", max_results=10):
    """Find items matching both rarity and buff type."""
    items_template = assets.templates["Item"]
    results = []
    
    for item in items_template.assets:
        # Check rarity
        if hasattr(item.Item, 'Rarity'):
            item_rarity = item.Item.Rarity()
            if item_rarity != rarity:
                continue
        else:
            continue
        
        # Check buffs
        for buff_ui in item.buff_ui:
            buff_text = buff_ui.text()
            
            if buff_keyword.lower() in buff_text.lower():
                results.append((item.text(), buff_text, buff_ui.value, item.guid))
                break
        
        if len(results) >= max_results:
            break
    
    return results

# Find legendary items with productivity buff
legendary_productivity = find_items_by_rarity_and_buff("Legendary", "Productivity", max_results=10)

print("Legendary Items with Productivity Buff:\n")
for name, buff_text, value, guid in legendary_productivity:
    print(f"  • {name:45s}")
    print(f"    {buff_text}: {value}")
    print()

Legendary Items with Productivity Buff:



## Next Steps

- `07_area_effects.ipynb` - Work with area/radius effects
- `04_buffs_and_effects.ipynb` - Learn more about buff structures